# NAKALA Complete Workflow Workshop

**🎯 Objective:** Learn to perform a complete cycle of NAKALA operations including data upload, collection creation, and curation analysis.

**📚 What you'll learn:**
- Upload research datasets to NAKALA repository
- Create and organize collections
- Perform data curation and quality analysis
- Generate comprehensive reports

**⏱️ Duration:** ~30 minutes

**🔧 Prerequisites:**
- NAKALA test account and API key
- Python environment with required packages
- Sample dataset (provided)

---

## 📋 Workshop Setup

### Step 1: Import Required Libraries

In [ ]:
# Method 1: Install nakala-client in development mode (recommended for local testing)
import subprocess
import sys
from pathlib import Path

print("🔧 Setting up nakala-client for workshop...")

try:
    # Try importing - if it fails, install in development mode
    import nakala_client
    print("✅ nakala-client is already installed")
    print(f"📍 Location: {nakala_client.__file__}")
except ImportError:
    print("📦 Installing nakala-client in development mode...")
    # Install from parent directory in development mode
    parent_dir = Path.cwd().parent
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(parent_dir)], 
                          capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ nakala-client installed successfully in development mode")
        import nakala_client
    else:
        print("⚠️ Development installation failed, trying fallback method...")
        print(f"Error: {result.stderr}")
        
        # Fallback: Add parent directory to path
        parent_dir = Path.cwd().parent
        if str(parent_dir) not in sys.path:
            sys.path.insert(0, str(parent_dir))
        print("✅ Added parent directory to Python path")

# Standard imports
import pandas as pd
import json
from datetime import datetime

# Try importing NAKALA client modules
try:
    from nakala_client.common.config import NakalaConfig
    from nakala_client.common.utils import NakalaCommonUtils
    import nakala_client.upload as upload_module
    import nakala_client.collection as collection_module  
    import nakala_client.curator as curator_module
    print("✅ NAKALA client modules imported successfully (PyPI-ready)")
    IMPORT_MODE = "pypi_ready"
except ImportError as e:
    print(f"⚠️ PyPI-style import failed: {e}")
    print("🔄 Using fallback import method...")
    # This will be removed once published to PyPI
    IMPORT_MODE = "fallback"

print(f"📁 Working directory: {Path.cwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"🔧 Import mode: {IMPORT_MODE}")

### Step 2: Configuration Setup

**🔑 IMPORTANT:** Replace `YOUR_API_KEY_HERE` with your actual NAKALA test API key!

In [ ]:
# Configuration
API_KEY = "YOUR_API_KEY_HERE"  # ⚠️ REPLACE WITH YOUR API KEY
API_URL = "https://apitest.nakala.fr"  # Test environment
WORKSHOP_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# 📖 PUBLIC TEST KEYS (for immediate workshop use):
# You can use any of these public test environment keys:
# - unakala1: 33170cfe-f53c-550b-5fb6-4814ce981293
# - unakala2: f41f5957-d396-3bb9-ce35-a4692773f636  
# - unakala3: aae99aba-476e-4ff2-2886-0aaf1bfa6fd2
# 
# Example: API_KEY = "f41f5957-d396-3bb9-ce35-a4692773f636"

# Paths
DATA_DIR = Path("data/sample_dataset")
OUTPUT_DIR = Path("outputs")
PARENT_DIR = Path("..")  # o-nakala-core directory

# Create output directory
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"🔧 Configuration completed")
print(f"📊 Workshop ID: {WORKSHOP_ID}")
print(f"🌐 API URL: {API_URL}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")

# Validate API key
if API_KEY == "YOUR_API_KEY_HERE":
    print("\n⚠️ WARNING: Please set your API key in the cell above!")
    print("💡 TIP: You can use one of the public test keys for immediate workshop use.")
    print("   See ../api/api_keys.md for available public test keys.")
else:
    print(f"✅ API Key configured (ends with: ...{API_KEY[-6:]})")

### Step 3: Verify Sample Dataset

Let's explore the sample dataset structure that we'll be working with.

In [ ]:
# Check dataset structure
print("📂 Sample Dataset Structure:")
print("=" * 40)

def show_tree(path, prefix="", max_depth=3, current_depth=0):
    if current_depth > max_depth:
        return
    
    if path.is_dir():
        items = sorted(path.iterdir())
        for i, item in enumerate(items):
            is_last = i == len(items) - 1
            current_prefix = "└── " if is_last else "├── "
            print(f"{prefix}{current_prefix}{item.name}")
            
            if item.is_dir():
                next_prefix = prefix + ("    " if is_last else "│   ")
                show_tree(item, next_prefix, max_depth, current_depth + 1)

show_tree(DATA_DIR)

# Show CSV configuration files
print("\n📄 Configuration Files:")
print("=" * 25)
csv_files = list(DATA_DIR.glob("*.csv"))
for csv_file in csv_files:
    print(f"📋 {csv_file.name} ({csv_file.stat().st_size} bytes)")

# Count total files
total_files = sum(1 for file in DATA_DIR.rglob("*") if file.is_file() and not file.name.endswith('.csv'))
print(f"\n📊 Total files to upload: {total_files}")

### Step 4: Preview Configuration Files

Let's examine the CSV files that define our upload and collection configuration.

In [ ]:
# Load and display folder data items configuration
data_items_csv = DATA_DIR / "folder_data_items.csv"
collections_csv = DATA_DIR / "folder_collections.csv"

print("📋 Data Items Configuration (folder_data_items.csv):")
print("=" * 60)
df_data = pd.read_csv(data_items_csv)
print(df_data.to_string(index=False))

print("\n\n📋 Collections Configuration (folder_collections.csv):")
print("=" * 60)
df_collections = pd.read_csv(collections_csv)
print(df_collections.to_string(index=False))

print(f"\n📊 Summary:")
print(f"   • {len(df_data)} data item groups to upload")
print(f"   • {len(df_collections)} collections to create")

---

## 🚀 Phase 1: Data Upload

In this phase, we'll upload our research files to NAKALA, organized by content type.

### Step 1: Validate Upload Configuration

Before uploading, let's validate our configuration to catch any issues early.

In [ ]:
# Smart validation using available import method
print("🔍 Validating upload configuration...")
print("=" * 60)

if IMPORT_MODE == "pypi_ready":
    print("🚀 Using direct API method (PyPI-ready)")
    try:
        # Method A: Direct API using current function-based structure
        # Note: This will evolve to class-based API for PyPI publication
        
        # Basic validation - check files exist and are readable
        print("📋 Validating dataset configuration...")
        
        if not data_items_csv.exists():
            print(f"❌ Dataset CSV not found: {data_items_csv}")
        elif not DATA_DIR.exists():
            print(f"❌ Data directory not found: {DATA_DIR}")
        else:
            # Count files for validation
            df_data = pd.read_csv(data_items_csv)
            total_datasets = len(df_data)
            
            total_files = 0
            for _, row in df_data.iterrows():
                folder_path = DATA_DIR / row['folder_path']
                if folder_path.exists():
                    files_in_folder = [f for f in folder_path.rglob("*") if f.is_file()]
                    total_files += len(files_in_folder)
            
            print("✅ Validation successful! Ready to upload.")
            print(f"📊 Found {total_files} files in {total_datasets} datasets")
            print(f"🔧 Using nakala_client modules for processing")
                
    except Exception as e:
        print(f"⚠️ Direct API validation failed: {e}")
        IMPORT_MODE = "fallback"  # Switch to fallback

if IMPORT_MODE == "fallback":
    print("🔄 Using CLI fallback method")
    # Method B: CLI subprocess using consolidated CLI commands
    import subprocess
    
    upload_cmd = [
        "nakala-upload",
        "--api-url", API_URL,
        "--api-key", API_KEY,
        "--dataset", str(data_items_csv.resolve()),  # Use absolute paths for data
        "--folder-config", str(data_items_csv.resolve()),
        "--mode", "folder",
        "--base-path", str(DATA_DIR.resolve()),
        "--validate-only"
    ]
    
    parent_dir = Path("..").resolve()
    
    print(f"📂 Working directory: {parent_dir}")
    print(f"Command: {' '.join(upload_cmd)}")
    
    try:
        # Execute from parent directory where CLI commands are available
        result = subprocess.run(upload_cmd, capture_output=True, text=True, cwd=parent_dir)
        print("📤 CLI Validation Output:")
        print(result.stdout)
        
        if result.stderr:
            print("⚠️ Validation Warnings/Errors:")
            print(result.stderr)
        
        if result.returncode == 0:
            print("✅ Validation successful! Ready to upload.")
        else:
            print(f"❌ Validation failed with return code: {result.returncode}")
            
    except Exception as cli_error:
        print(f"❌ CLI method also failed: {cli_error}")
        print("\n💡 Troubleshooting:")
        print("   • Ensure nakala-client is installed: pip install -e ../")
        print("   • Check CLI commands are available: nakala-upload --help")

print(f"\n🔧 Method used: {IMPORT_MODE}")
print("📝 Note: After PyPI publication, this will use a clean class-based API")

### Step 2: Execute Data Upload

Now let's perform the actual upload to NAKALA!

In [ ]:
# Execute upload using the configured method
upload_output = OUTPUT_DIR / f"upload_results_{WORKSHOP_ID}.csv"

if IMPORT_MODE == "pypi_ready":
    print("🚀 Starting data upload using direct API...")
    print(f"📁 Output file: {upload_output}")
    print("\n" + "=" * 50)
    
    try:
        # Future: This will use a clean API like:
        # uploader = NakalaUploader(config)
        # result = uploader.upload_folder(dataset_csv=data_items_csv, base_path=DATA_DIR)
        
        # For now, we'll use the CLI method even in PyPI mode
        print("🔄 Using CLI method for actual upload")
        IMPORT_MODE = "fallback"
        
    except Exception as e:
        print(f"⚠️ Direct API upload failed: {e}")
        IMPORT_MODE = "fallback"

if IMPORT_MODE == "fallback":
    print("🚀 Starting data upload using CLI method...")
    print(f"📁 Output file: {upload_output}")
    print("\n" + "=" * 50)
    
    # Actual upload command using consolidated CLI
    upload_cmd = [
        "nakala-upload",
        "--api-url", API_URL,
        "--api-key", API_KEY,
        "--dataset", str(data_items_csv),
        "--folder-config", str(data_items_csv),
        "--mode", "folder",
        "--base-path", str(DATA_DIR),
        "--output", str(upload_output)
    ]
    
    parent_dir = Path("..").resolve()
    
    try:
        result = subprocess.run(upload_cmd, capture_output=True, text=True, cwd=parent_dir)
        
        print("📤 Upload Output:")
        print(result.stdout)
        
        if result.stderr:
            print("\n⚠️ Upload Warnings/Errors:")
            print(result.stderr)
        
        if result.returncode == 0:
            print("\n✅ Upload completed successfully!")
        else:
            print(f"\n❌ Upload failed with return code: {result.returncode}")
            
    except Exception as e:
        print(f"❌ Error during upload: {e}")

### Step 3: Analyze Upload Results

Let's examine what was successfully uploaded.

In [ ]:
# Load and analyze upload results
if upload_output.exists():
    print("📊 Upload Results Analysis:")
    print("=" * 40)
    
    df_upload = pd.read_csv(upload_output)
    
    # Display results
    print(f"📈 Total data items created: {len(df_upload)}")
    print(f"✅ Successful uploads: {len(df_upload[df_upload['status'] == 'OK'])}")
    
    if len(df_upload[df_upload['status'] != 'OK']) > 0:
        print(f"❌ Failed uploads: {len(df_upload[df_upload['status'] != 'OK'])}")
    
    print("\n📋 Uploaded Data Items:")
    for _, row in df_upload.iterrows():
        files_count = len(row['files'].split(','))
        print(f"   🔗 {row['identifier']} - {row['title']} ({files_count} files)")
    
    print("\n📄 Detailed Results:")
    display_cols = ['identifier', 'title', 'status']
    print(df_upload[display_cols].to_string(index=False))
    
else:
    print("❌ Upload output file not found. Upload may have failed.")

---

## 📚 Phase 2: Collection Creation

Now we'll organize our uploaded data into thematic collections.

### Step 1: Create Collections

Using our folder collections configuration, we'll create organized collections.

In [ ]:
# Output file for collection results
collection_output = OUTPUT_DIR / f"collections_results_{WORKSHOP_ID}.csv"

# Collection creation command using consolidated CLI
collection_cmd = [
    "nakala-collection",
    "--api-url", API_URL,
    "--api-key", API_KEY,
    "--from-folder-collections", str(collections_csv),
    "--from-upload-output", str(upload_output),
    "--collection-report", str(collection_output)
]

print("📚 Creating NAKALA collections...")
print(f"📁 Output file: {collection_output}")
print("\n" + "=" * 50)

parent_dir = Path("..").resolve()

try:
    result = subprocess.run(collection_cmd, capture_output=True, text=True, cwd=parent_dir)
    
    print("📚 Collection Creation Output:")
    print(result.stdout)
    
    if result.stderr:
        print("\n⚠️ Collection Warnings/Errors:")
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✅ Collections created successfully!")
    else:
        print(f"\n❌ Collection creation failed with return code: {result.returncode}")
        
except Exception as e:
    print(f"❌ Error during collection creation: {e}")

### Step 2: Analyze Collection Results

Let's examine the collections that were created.

In [ ]:
# Load and analyze collection results
if collection_output.exists():
    print("📊 Collection Results Analysis:")
    print("=" * 40)
    
    df_collections = pd.read_csv(collection_output)
    
    # Display results
    print(f"📈 Total collections created: {len(df_collections)}")
    successful = len(df_collections[df_collections['creation_status'] == 'SUCCESS'])
    print(f"✅ Successful creations: {successful}")
    
    if len(df_collections[df_collections['creation_status'] != 'SUCCESS']) > 0:
        failed = len(df_collections[df_collections['creation_status'] != 'SUCCESS'])
        print(f"❌ Failed creations: {failed}")
    
    print("\n📋 Created Collections:")
    for _, row in df_collections.iterrows():
        data_items = row['data_items_ids'].split(';') if pd.notna(row['data_items_ids']) else []
        print(f"   🗂️ {row['collection_id']} - {row['collection_title']}")
        print(f"      📊 {row['data_items_count']} data items | Status: {row['status']}")
    
    print("\n📄 Detailed Results:")
    display_cols = ['collection_id', 'collection_title', 'status', 'data_items_count', 'creation_status']
    print(df_collections[display_cols].to_string(index=False))
    
else:
    print("❌ Collection output file not found. Collection creation may have failed.")

---

## 🔍 Phase 3: Data Curation & Quality Analysis

Now we'll perform curation analysis to assess data quality and generate recommendations.

### Step 1: Data Items Curation Analysis

Let's analyze the quality of our uploaded data items.

In [ ]:
# Output file for data curation report
data_curation_output = OUTPUT_DIR / f"data_curation_report_{WORKSHOP_ID}.json"

# Data curation command using consolidated CLI
data_curation_cmd = [
    "nakala-curator",
    "--api-url", API_URL,
    "--api-key", API_KEY,
    "--quality-report",
    "--output", str(data_curation_output)
]

print("🔍 Performing data curation analysis...")
print(f"📁 Output file: {data_curation_output}")
print("\n" + "=" * 50)

parent_dir = Path("..").resolve()

try:
    result = subprocess.run(data_curation_cmd, capture_output=True, text=True, cwd=parent_dir)
    
    print("🔍 Data Curation Output:")
    print(result.stdout)
    
    if result.stderr:
        print("\n⚠️ Curation Warnings/Errors:")
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✅ Data curation analysis completed!")
    else:
        print(f"\n❌ Data curation failed with return code: {result.returncode}")
        
except Exception as e:
    print(f"❌ Error during data curation: {e}")

### Step 2: Collection-Focused Curation Analysis

Let's also perform a collection-specific analysis.

In [ ]:
# Output file for collection curation report
collection_curation_output = OUTPUT_DIR / f"collection_curation_report_{WORKSHOP_ID}.json"

# Collection curation command using consolidated CLI
collection_curation_cmd = [
    "nakala-curator",
    "--api-url", API_URL,
    "--api-key", API_KEY,
    "--quality-report",
    "--scope", "collections",
    "--output", str(collection_curation_output)
]

print("🔍 Performing collection curation analysis...")
print(f"📁 Output file: {collection_curation_output}")
print("\n" + "=" * 50)

parent_dir = Path("..").resolve()

try:
    result = subprocess.run(collection_curation_cmd, capture_output=True, text=True, cwd=parent_dir)
    
    print("🔍 Collection Curation Output:")
    print(result.stdout)
    
    if result.stderr:
        print("\n⚠️ Curation Warnings/Errors:")
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✅ Collection curation analysis completed!")
    else:
        print(f"\n❌ Collection curation failed with return code: {result.returncode}")
        
except Exception as e:
    print(f"❌ Error during collection curation: {e}")

### Step 3: Analyze Curation Reports

Let's examine the generated curation reports to understand data quality.

In [ ]:
# Load and analyze curation reports
print("📊 Curation Analysis Results:")
print("=" * 40)

# Data curation report
if data_curation_output.exists():
    print("\n🔍 Data Items Quality Report:")
    print("-" * 30)
    
    with open(data_curation_output, 'r') as f:
        data_report = json.load(f)
    
    summary = data_report.get('summary', {})
    print(f"   📊 Total Collections: {summary.get('total_collections', 0)}")
    print(f"   📊 Total Datasets: {summary.get('total_datasets', 0)}")
    print(f"   📊 Overall Quality Score: {data_report.get('overall_quality_score', 0):.1f}")
    
    recommendations = data_report.get('recommendations', [])
    if recommendations:
        print("\n   💡 Recommendations:")
        for i, rec in enumerate(recommendations, 1):
            print(f"      {i}. {rec}")
    
    print(f"\n   📅 Generated: {data_report.get('generated_at', 'Unknown')}")
    
else:
    print("❌ Data curation report not found.")

# Collection curation report
if collection_curation_output.exists():
    print("\n🗂️ Collections Quality Report:")
    print("-" * 30)
    
    with open(collection_curation_output, 'r') as f:
        collection_report = json.load(f)
    
    summary = collection_report.get('summary', {})
    print(f"   📊 Total Collections: {summary.get('total_collections', 0)}")
    print(f"   📊 Total Datasets: {summary.get('total_datasets', 0)}")
    print(f"   📊 Overall Quality Score: {collection_report.get('overall_quality_score', 0):.1f}")
    
    recommendations = collection_report.get('recommendations', [])
    if recommendations:
        print("\n   💡 Recommendations:")
        for i, rec in enumerate(recommendations, 1):
            print(f"      {i}. {rec}")
    
    print(f"\n   📅 Generated: {collection_report.get('generated_at', 'Unknown')}")
    
else:
    print("❌ Collection curation report not found.")

---

## 📋 Phase 4: Comprehensive Workshop Report

Let's generate a final comprehensive report of our workshop activities.

In [ ]:
# Generate comprehensive workshop report
workshop_report = OUTPUT_DIR / f"workshop_report_{WORKSHOP_ID}.md"

report_content = f"""# NAKALA Workshop Report

**🎯 Workshop ID:** {WORKSHOP_ID}
**📅 Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**🌐 API Environment:** {API_URL}
**🔑 API Key:** ...{API_KEY[-6:] if API_KEY != 'YOUR_API_KEY_HERE' else 'NOT_SET'}

## 📊 Workshop Summary

### Phase 1: Data Upload
"""

# Add upload results if available
if upload_output.exists():
    df_upload = pd.read_csv(upload_output)
    successful_uploads = len(df_upload[df_upload['status'] == 'OK'])
    total_uploads = len(df_upload)
    
    report_content += f"""
- **Status:** ✅ Completed
- **Data Items Created:** {total_uploads}
- **Success Rate:** {successful_uploads}/{total_uploads} ({(successful_uploads/total_uploads*100):.1f}%)
- **Output File:** `{upload_output.name}`

#### Uploaded Data Items:
"""
    for _, row in df_upload.iterrows():
        files_count = len(row['files'].split(','))
        status_emoji = "✅" if row['status'] == 'OK' else "❌"
        report_content += f"\n- {status_emoji} **{row['identifier']}** - {row['title']} ({files_count} files)"
else:
    report_content += "\n- **Status:** ❌ Failed or not completed"

report_content += "\n\n### Phase 2: Collection Creation"

# Add collection results if available
if collection_output.exists():
    df_collections = pd.read_csv(collection_output)
    successful_collections = len(df_collections[df_collections['creation_status'] == 'SUCCESS'])
    total_collections = len(df_collections)
    
    report_content += f"""
- **Status:** ✅ Completed
- **Collections Created:** {total_collections}
- **Success Rate:** {successful_collections}/{total_collections} ({(successful_collections/total_collections*100):.1f}%)
- **Output File:** `{collection_output.name}`

#### Created Collections:
"""
    for _, row in df_collections.iterrows():
        status_emoji = "✅" if row['creation_status'] == 'SUCCESS' else "❌"
        report_content += f"\n- {status_emoji} **{row['collection_id']}** - {row['collection_title']} ({row['data_items_count']} items)"
else:
    report_content += "\n- **Status:** ❌ Failed or not completed"

report_content += "\n\n### Phase 3: Curation Analysis"

# Add curation results if available
curation_files = [data_curation_output, collection_curation_output]
curation_names = ["Data Items", "Collections"]
completed_curations = 0

for i, (file, name) in enumerate(zip(curation_files, curation_names)):
    if file.exists():
        completed_curations += 1
        with open(file, 'r') as f:
            curation_data = json.load(f)
        quality_score = curation_data.get('overall_quality_score', 0)
        report_content += f"\n- **{name} Analysis:** ✅ Completed (Quality Score: {quality_score:.1f})"
    else:
        report_content += f"\n- **{name} Analysis:** ❌ Failed or not completed"

if completed_curations > 0:
    report_content += f"\n- **Overall Status:** ✅ {completed_curations}/2 analyses completed"
else:
    report_content += "\n- **Overall Status:** ❌ No analyses completed"

# Add output files summary
report_content += f"""

## 📁 Generated Files

All workshop outputs are stored in: `{OUTPUT_DIR}/`

### Output Files:
"""

output_files = list(OUTPUT_DIR.glob(f"*{WORKSHOP_ID}*"))
for file in output_files:
    size = file.stat().st_size
    report_content += f"\n- 📄 `{file.name}` ({size:,} bytes)"

# Add learning outcomes
report_content += f"""

## 🎓 Learning Outcomes Achieved

In this workshop, you successfully:

1. **✅ Configured NAKALA API Access** - Set up authentication and environment
2. **✅ Explored Dataset Structure** - Understood folder-based organization
3. **✅ Validated Configuration** - Used validation mode before actual operations
4. **✅ Uploaded Research Data** - Created {len(df_upload) if upload_output.exists() else 0} data items with multiple file types
5. **✅ Created Collections** - Organized data into {len(df_collections) if collection_output.exists() else 0} thematic collections
6. **✅ Performed Curation** - Analyzed data quality and received recommendations
7. **✅ Generated Reports** - Created comprehensive documentation of the workflow

## 🚀 Next Steps

Now that you've completed the workshop:

1. **Explore Your Data** - Visit {API_URL} to view your uploaded items
2. **Enhance Metadata** - Use the curation recommendations to improve data quality
3. **Share Collections** - Consider making collections public when ready
4. **Scale Up** - Apply these techniques to your own research datasets
5. **Automate Workflows** - Use the CLI tools in your research pipelines

## 📚 Additional Resources

- **NAKALA Documentation:** https://documentation.nakala.fr/
- **API Reference:** https://api.nakala.fr/
- **O-Nakala Tools:** Check the parent directory for more advanced features

---

**🎉 Congratulations on completing the NAKALA Complete Workflow Workshop!**
"""

# Write the report
with open(workshop_report, 'w') as f:
    f.write(report_content)

print(f"📋 Workshop report generated: {workshop_report}")
print(f"📁 Total output files: {len(list(OUTPUT_DIR.glob('*')))}")
print("\n✅ Workshop completed successfully!")

## 🎉 Workshop Summary

Congratulations! You've successfully completed the NAKALA Complete Workflow Workshop.

### What You Accomplished:

1. **🔧 Setup & Configuration** - Configured API access and validated environment
2. **📤 Data Upload** - Uploaded research files organized by content type
3. **📚 Collection Creation** - Organized data into thematic collections
4. **🔍 Curation Analysis** - Performed quality assessment and received recommendations
5. **📋 Report Generation** - Created comprehensive documentation

### Key Skills Learned:

- Using NAKALA v2.0 CLI tools
- Folder-based dataset organization
- Collection management strategies
- Data quality assessment techniques
- Workflow automation with Python

### Next Steps:

1. **Explore** your uploaded data in the NAKALA interface
2. **Apply** these techniques to your own research datasets
3. **Enhance** metadata based on curation recommendations
4. **Share** your collections with the research community

---

**📁 All workshop outputs are saved in the `outputs/` directory.**

**🎓 You're now ready to manage research data effectively with NAKALA!**